# 05 — Final Analysis, Interpretability and Business Synthesis

**Objectives:** Review selected ResNet18 decisions with Grad-CAM and intermediate activations; reconcile results from all four O/S/T/ST regimes; integrate the new progressive T/ST proof of principle; and formulate evidence-limited operational recommendations.

Interpretability images are descriptive, not causal evidence. Historical outcomes are retained as accepted evidence. The separate progressive T/ST study was completed on **3 November 2026**; its measured results are displayed from either local output tables or the committed eight-run summary, and the limitations of both protocols remain explicit.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image as IPyImage
import src as mr
ROOT=Path.cwd().resolve()
settings=mr.load_settings(ROOT)
device=mr.setup(settings['seed'])
# Setting RERUN_HISTORICAL_MODELS = False shows archived Grad-CAM, activation
# and held-out score evidence; it does NOT require old model checkpoints.
# Setting RERUN_HISTORICAL_MODELS = True regenerates selected analyses from
# newly trained ResNet18/Scratch CNN checkpoints in generated/outputs/.
RERUN_HISTORICAL_MODELS = False
print('Historical analysis in report-first mode; no original model retraining.')


## Grad-CAM 4×4 preview

The following code loads each selected **frozen checkpoint** and the actual images corresponding to the test predictions, calculates layer-4 Grad-CAM, overlays it on the image, and saves the selection registry. Heat-map colours show positive normalized attribution magnitude to the nominated class; they do not establish that a physical feature caused the prediction. The archived Grad-CAM panel is shown by default; optional checkpoint-dependent regeneration remains available after a complete rerun.

In [ ]:
if RERUN_HISTORICAL_MODELS:
    camfile=mr.plot_cam_matrix(settings,device,Path(settings['out'])/'figures'/'resnet18_gradcam_4x4.png')
    display(IPyImage(filename=str(camfile)))
    reg=pd.read_csv(Path(settings['out'])/'figures'/'gradcam_16_panel_selection.csv')
    display(reg[['regime','role','found','sample_id']])
else:
    display(IPyImage(filename=str(ROOT/'figures'/'resnet18_gradcam_4x4.jpg')))
    print('Original Grad-CAM panel shown; it is not regenerated from a new checkpoint.')


## Full case registry and intermediate activations

The full registry targets 21 distinct case choices when the held-out populations contain the required examples. We retain the selection provenance and export one mean-absolute activation map per case at each of four ResNet stages. The 21-case analysis is retained as accepted historical evidence, and optional checkpoint-based regeneration is a separate full-rerun mode. When no specific role exists, the registry records the available cases rather than manufacturing absent examples.

In [ ]:
if RERUN_HISTORICAL_MODELS:
    registry,activation=mr.generate_activation_diagnostics(settings,device,Path(settings['out'])/'figures'/'resnet18_intermediate_activations.png')
    display(registry.head(10));display(activation.head(10))
    display(IPyImage(filename=str(Path(settings['out'])/'figures'/'resnet18_intermediate_activations.png')))
else:
    print("Original 21-case/activation evidence accepted from the archived submission.")


## Technical conclusion and limits

The valid final interpretation uses only actual run outputs: independently fitted models, held-out predictions, source-bootstrap intervals, frame-level error records and class-directed visualizations. The four regimes are separate experimental designs, **not sequential fine-tuning**. A high held-out score does not guarantee robust new-source deployment. Cases and heatmaps illustrate hypotheses and failure patterns but cannot demonstrate their mechanisms causally. Research limitations include reconstruction of source IDs from filenames, correlated frames, recording coverage, optimizer randomness, and potentially scarce confident-error examples.

**Reproduction check:** notebooks 01–05 must execute in order from the same project directory with the correct dataset; the output folder contains split hashes, checkpoints, trial histories, per-frame predictions, figures and selection registries. No reported score is asserted to equal the author's published number. The notebook reader should compare their own measured outputs to any reported benchmark instead.

In [ ]:
if RERUN_HISTORICAL_MODELS:
    summary=[]
    for regime in mr.REGIMES:
        res=pd.read_csv(Path(settings['out'])/'resnet18'/regime/'heldout_test_predictions.csv')
        scratch=pd.read_csv(Path(settings['out'])/'scratch'/regime/'heldout_test_predictions.csv')
        summary.append({'regime':regime,'scratch_test_macro_f1':mr.performance(scratch)['macro_f1'],
                        'resnet18_test_macro_f1':mr.performance(res)['macro_f1']})
    summary=pd.DataFrame(summary)
    display(summary)
    summary.to_csv(Path(settings['out'])/'final_recomputed_summary.csv',index=False)
    print('Complete: figures and results above were computed from this run, not pasted from historic tables.')
else:
    print("Historical O/S/T/ST metrics and final interpretations loaded from submitted tables below.")


## Original interpretability and synthesis evidence — reference only

This source-derived material is retained for provenance, not used to select checkpoints or tune new test outcomes.

In [ ]:
historical_activation=mr.read_archived_result_table(ROOT/'submission'/'results'/'activation_summary.csv')
historical_conclusions=mr.read_archived_result_table(ROOT/'submission'/'results'/'final_scientific_conclusions.csv')
display(historical_activation.head(15))
display(historical_conclusions)

## Progressive temporal findings — completed supplementary synthesis

This section reads the fresh results created in Notebook 04. Compare overall remaining-future Macro F1 and flip recall, matched-final-10% performance, and supported horizon windows separately for T and ST. Differences between T and ST may arise from source distribution changes and must not automatically be interpreted as temporal drift.

The recorded experiment is a **retrospective proof of principle**. It has neither verified capture timestamps nor a mechanism to automatically obtain new supervision in deployment. Grad-CAM regions are associations, not proof that a particular feature causes a model's predictions.


In [ ]:
temporal_root=Path(settings['out'])/'progressive_temporal'
if (temporal_root/'overall_results.csv').exists() and (temporal_root/'horizon_results.csv').exists():
    overall=pd.read_csv(temporal_root/'overall_results.csv')
    horizons=pd.read_csv(temporal_root/'horizon_results.csv')
    display(overall[['regime','coverage_percent','n_train','n_validation','n_future',
                     'future_macro_f1','future_flip_recall','false_negatives',
                     'matched_macro_f1','matched_flip_recall']])
    display(horizons.loc[horizons.reportable==True,
        ['regime','coverage_percent','start_horizon','end_horizon',
         'n_frames','n_flip','n_notflip','macro_f1','flip_recall']])
else:
    summary_file=ROOT/'submission/results/progressive_temporal_completed_8_runs.csv'
    if summary_file.is_file():
        finished=pd.read_csv(summary_file)
        assert len(finished)==8, 'Expected eight completed progressive records'
        display(finished[['regime','coverage_percent','n_train','n_validation','n_future',
            'future_macro_f1','future_flip_recall','false_negatives',
            'matched_macro_f1','matched_flip_recall']])
        print('Completed supplementary run summary from the 2026-11-03 archived report; no new predictions regenerated.')
    else:
        print('No local progressive outputs or committed completed-run summary found.')


## Final interpretation and deployment recommendation — evidence-based review

1. **Reliability:** State which regime provides the most relevant estimate for new books or camera contexts, and quantify uncertainty.
2. **Missed page turns:** Compare flip recall and false negatives, especially confident errors that would interrupt hands-free document reading.
3. **Progressive updating:** Report whether the newly executed T and ST coverage curves improve enough to justify experimenting with retraining from newly labelled frames. Distinguish whole-future from near/far-position findings.
4. **Model size and latency:** Prefer ResNet18 if its robustness benefit justifies the measured compute; evaluate the smaller CNN and grayscale options only against real hardware/runtime constraints.
5. **Pilot criteria:** Collect genuinely independent recordings, timestamps, page-turn event annotations, latency and false-negative tolerances before deployment. Do not claim that these notebook experiments establish continuous online learning.

The final written conclusion must be populated **only after reviewing the executed numbers**, with unsupported alternatives presented as hypotheses rather than causal explanations.
